# ПЗ 4 — Извлечение аудио и Whisper

**Задача:** взять видео из Drive, извлечь аудио, транскрибировать через Whisper.

**Стек:** `moviepy`, `openai-whisper`, `ffmpeg`

In [ ]:
!pip install moviepy openai-whisper -q
!apt-get install -y ffmpeg -q

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

BASE_DRIVE  = '/content/drive/MyDrive/cv-frames'
VIDEO_DIR   = f'{BASE_DRIVE}/видио'
RESULTS_DIR = f'{BASE_DRIVE}/результаты'

os.makedirs(RESULTS_DIR, exist_ok=True)

# Выбрать видео из Drive
videos = os.listdir(VIDEO_DIR)
if not videos:
    print('❌ Видео не найдены в Drive — сначала запустите ПЗ 2')
else:
    print('Видео в Drive:')
    for i, v in enumerate(videos):
        print(f'  [{i}] {v}')
    video_path = f'{VIDEO_DIR}/{videos[0]}'  # берём первое
    print(f'\n✅ Используем: {video_path}')

In [ ]:
from moviepy.editor import VideoFileClip

AUDIO_PATH = '/content/audio.wav'
clip = VideoFileClip(video_path)
clip.audio.write_audiofile(AUDIO_PATH, verbose=False, logger=None)
clip.close()
print(f'✅ Аудио извлечено: {AUDIO_PATH}')

In [ ]:
import whisper

model = whisper.load_model('base')  # base / medium / large
result = model.transcribe(AUDIO_PATH, language='ru', verbose=False)
print('✅ Транскрипция завершена')
print('Превью:', result['text'][:300])

In [ ]:
import pandas as pd

rows = [{'start': round(s['start'],2), 'end': round(s['end'],2), 'text': s['text'].strip()}
        for s in result['segments']]
df = pd.DataFrame(rows)

OUT = f'{RESULTS_DIR}/whisper_transcript.csv'
df.to_csv(OUT, index=False, encoding='utf-8-sig')
print(f'✅ Сохранено сегментов: {len(df)} → {OUT}')
df.head(10)